# Telco Customer Churn Prediction  
### Demo of a Supervised Learning System Application

This notebook demonstrates a supervised learning application that predicts whether a teleco customer is likely to churn. The model is trained using the Telco Customer Churn dataset and applies a Decision Tree classifier to learn patterns associated with customer retention and churn behavior.

The goal of this demo is to illustrate how labeled historical data can be used to build a predictive model that helps organizations identify customers at risk of leaving their service. Such insights can support retention strategies and improve decision making for marketing and customer service teams.

## Imports and Setup

This section imports the Python libraries required for the project. These libraries support data manipulation, machine learning, and model evaluation.

- **pandas** and **numpy** are used for data processing and numerical operations.  
- **scikit-learn** provides tools for training the Decision Tree model, preprocessing features, splitting the dataset, and computing evaluation metrics.

These libraries form the core environment for building and testing the supervised learning model.

In [177]:
import textwrap
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

## Load the Dataset

In this step, the Telco Customer Churn dataset is loaded into a pandas DataFrame. The dataset contains customer demographic information, account details, service subscriptions, and billing data, along with a labeled churn indicator.

Each row represents a customer record, and the **Churn** column indicates whether the customer left the service (Yes) or remained (No).

In [178]:
df = pd.read_csv("./data/telco-customer-churn.csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [179]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

## Basic Inspection and Data Cleaning

Before training the model, the dataset is inspected and cleaned to ensure that the data is in a suitable format for analysis.

The following preprocessing steps are performed:

- Convert **TotalCharges** to numeric format, since it may be read as text during import.
- Remove records with missing values that cannot be converted properly.
- Drop the **customerID** column because it is a unique identifier and does not contain predictive information for churn behavior.

These steps help ensure that the dataset is consistent and ready for machine learning.

In [180]:
# totalCharges sometimes loads as text, convert safely
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Drop rows where TotalCharges is missing after conversion
df = df.dropna(subset=["TotalCharges"]).reset_index(drop=True)

# customerID is an identifier, not predictive
df = df.drop(columns=["customerID"])

df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Define Features (X) and Target Variable (Y)

The dataset is separated into input features and the target variable.

- **X (features)** contains all customer attributes used for prediction, such as service usage, billing information, and account details.
- **Y (target variable)** represents the churn outcome.

The churn labels are converted into numeric form:
- 0 = No churn
- 1 = Churn

This numeric representation allows the model to perform binary classification.

In [181]:
target_col = "Churn"
x = df.drop(columns=[target_col])
y = df[target_col].map({"No": 0, "Yes": 1})  # 0 = No churn, 1 = Yes churn

x.shape, y.shape

((7032, 19), (7032,))

## Check Class Distribution

Before training the model, the distribution of churn and non-churn customers is examined. Understanding class distribution is important because machine learning models can be affected by class imbalance.

If one class appears significantly more often than the other, evaluation metrics beyond simple accuracy may be required to properly assess model performance.

In [182]:
class_counts = y.value_counts().rename(index={0:"No", 1:"Yes"})
class_pct = (y.value_counts(normalize=True)*100).rename(index={0:"No", 1:"Yes"})

print("Class distribution (counts):")
print(class_counts)
print("\nClass distribution (%):")
print(class_pct.round(2))

Class distribution (counts):
Churn
No     5163
Yes    1869
Name: count, dtype: int64

Class distribution (%):
Churn
No     73.42
Yes    26.58
Name: proportion, dtype: float64


## Split the Dataset into Training and Testing Sets

The dataset is divided into two parts using the `train_test_split` function from scikit-learn.

- **80% of the data** is used for training the model.
- **20% of the data** is reserved for testing.

Stratified sampling is applied to maintain the same churn distribution in both subsets. The model learns patterns from the training data and is evaluated using the unseen test data.

In [183]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # keeps churn ratio similar in train/ test
)

x_train.shape, x_test.shape

((5625, 19), (1407, 19))

## Build the Preprocessing and Modeling Pipeline

This step creates a machine learning pipeline that combines data preprocessing and model training into a single workflow.

Categorical variables such as contract type, internet service, and payment method are transformed using **One-Hot Encoding** so that they can be used by the model. Numerical features are passed through without modification.

A **Decision Tree Classifier** is then applied to learn patterns in the data and predict whether a customer is likely to churn.

In [184]:
# Identify column types
cat_cols = x.select_dtypes(include=["object", "str"]).columns.tolist()
num_cols = x.select_dtypes(exclude=["object", "str"]).columns.tolist()

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ]
)

model = DecisionTreeClassifier(
    random_state=42,
    max_depth=6  # keep it simple, reduces overfitting for demo
)

pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", model)
])

pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers cont

## Train the Model

The Decision Tree model is trained using the training dataset. During this process, the model analyzes relationships between customer attributes and churn outcomes in order to learn decision rules that separate churn and non-churn customers.

Once training is complete, the model can be used to make predictions on new or unseen data.

In [185]:
pipe.fit(x_train, y_train)
print("Model trained.")

Model trained.


## Make Predictions on the Test Set

After the model has been trained, predictions are generated for the test dataset.

Two outputs are produced:

- **Predicted class labels** indicating whether the model predicts churn or no churn.
- **Predicted probabilities**, which represent the likelihood that a customer will churn.

These predictions are then used to evaluate the model's performance.

In [186]:
y_pred = pipe.predict(x_test)
y_proba = pipe.predict_proba(x_test)[:, 1]  # probability of churn=1

y_pred[:10], y_proba[:10]

(array([0, 1, 0, 0, 0, 0, 0, 0, 1, 0]),
 array([0.01401869, 0.59259259, 0.01401869, 0.12371134, 0.09134615,
        0.42424242, 0.01918465, 0.08898305, 0.55792683, 0.00213675]))

## Confusion Matrix

A confusion matrix is used to analyze the model's classification results.

It summarizes prediction outcomes in four categories:

- **True Positive (TP):** correctly predicted churn
- **True Negative (TN):** correctly predicted non-churn
- **False Positive (FP):** predicted churn but customer stayed
- **False Negative (FN):** predicted non-churn but customer churned

This matrix provides a detailed view of how well the model distinguishes between churn and non-churn customers.

In [187]:
cm = confusion_matrix(y_test, y_pred)  # [[TN, FP],[FN, TP]]
tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix [[TN, FP],[FN, TP]]:")
print(cm)
print("\nBreakdown:")
print(f"TP (True Positive)  = {tp}")
print(f"TN (True Negative)  = {tn}")
print(f"FP (False Positive) = {fp}")
print(f"FN (False Negative) = {fn}")

Confusion Matrix [[TN, FP],[FN, TP]]:
[[903 130]
 [165 209]]

Breakdown:
TP (True Positive)  = 209
TN (True Negative)  = 903
FP (False Positive) = 130
FN (False Negative) = 165


## Evaluation Metrics

Several evaluation metrics are calculated to measure model performance:

- **Accuracy:** overall proportion of correct predictions
- **Balanced Accuracy:** accounts for class imbalance
- **Precision:** how many predicted churn cases were correct
- **Recall:** how many actual churn cases were detected
- **F1 Score:** balance between precision and recall
- **ROC-AUC:** ability of the model to distinguish between churn and non-churn customers

Together, these metrics provide a comprehensive evaluation of the classification model.

In [188]:
acc = accuracy_score(y_test, y_pred)
bacc = balanced_accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
auc = roc_auc_score(y_test, y_proba)

print(f"Accuracy:          {acc:.4f}")
print(f"Balanced Accuracy: {bacc:.4f}")
print(f"Precision:         {prec:.4f}")
print(f"Recall:            {rec:.4f}")
print(f"F1-score:          {f1:.4f}")
print(f"ROC-AUC:           {auc:.4f}")

Accuracy:          0.7903
Balanced Accuracy: 0.7165
Precision:         0.6165
Recall:            0.5588
F1-score:          0.5863
ROC-AUC:           0.8232


In [189]:
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["No Churn (0)", "Churn (1)"]))

Classification Report:
              precision    recall  f1-score   support

No Churn (0)       0.85      0.87      0.86      1033
   Churn (1)       0.62      0.56      0.59       374

    accuracy                           0.79      1407
   macro avg       0.73      0.72      0.72      1407
weighted avg       0.78      0.79      0.79      1407



## Feature Importance Analysis

To better understand how the model makes decisions, feature importance values are extracted from the trained Decision Tree. These values indicate how strongly each feature contributes to reducing classification error during training.

Features with higher importance scores play a larger role in predicting customer churn. This analysis helps identify the key factors that influence churn behavior, such as contract type, service charges, and customer tenure.

In [190]:
# Grab fitted parts
pre = pipe.named_steps["preprocess"]
tree = pipe.named_steps["model"]

cat_cols = x.select_dtypes(include=["object", "str"]).columns.tolist()
num_cols = x.select_dtypes(exclude=["object", "str"]).columns.tolist()

ohe = pre.named_transformers_["cat"]
feature_names = list(ohe.get_feature_names_out(cat_cols)) + num_cols

importances = tree.feature_importances_
fi = pd.DataFrame({"feature": feature_names, "importance": importances}).sort_values("importance", ascending=False)

fi.head(20)

,feature,importance
32,Contract_Month-to-month,0.480727
44,TotalCharges,0.141078
12,InternetService_Fiber optic,0.139163
42,tenure,0.065969
43,MonthlyCharges,0.043349
23,TechSupport_No,0.025589
39,PaymentMethod_Electronic check,0.018780
41,SeniorCitizen,0.011561
33,Contract_One year,0.010894
31,StreamingMovies_Yes,0.009814


## Example Predictions

Finally, three sample customers from the test dataset are evaluated to demonstrate how the model performs on individual records.

For each example, the notebook displays:
- The input customer attributes
- The predicted churn outcome
- The predicted probability of churn
- The actual churn label

These examples help illustrate how the trained model interprets real customer profiles and generates predictions.

In [191]:
# Pick 3 test examples and show model output
sample_idx = x_test.sample(3, random_state=7).index
print(sample_idx)
sample_x = x.loc[sample_idx]
sample_y_true = y.loc[sample_idx]

sample_pred = pipe.predict(sample_x)
sample_proba = pipe.predict_proba(sample_x)[:, 1]

result = sample_x.copy()
result["True_Churn"] = sample_y_true.values
result["Predicted_Churn"] = sample_pred
result["Predicted_Prob_Churn"] = np.round(sample_proba, 3)

result

Index([2228, 2580, 653], dtype='int64')


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,True_Churn,Predicted_Churn,Predicted_Prob_Churn
2228,Male,0,No,No,54,Yes,No,DSL,Yes,Yes,...,Yes,No,Month-to-month,Yes,Mailed check,69.90,3883.3,0,0,0.089
2580,Female,0,No,No,37,Yes,No,No,No internet service,No internet service,...,No internet service,No internet service,Two year,No,Mailed check,19.80,813.3,0,0,0.014
653,Female,1,No,No,8,Yes,No,No,No internet service,No internet service,...,No internet service,No internet service,Month-to-month,No,Electronic check,19.65,164.3,1,0,0.143


### Example 1 — Customer Profile (Index 3154)

This customer has few characteristics associated with higher churn risk. The customer has a **short tenure of 3 months**, a **month-to-month contract**, and **relatively high monthly charges ($94.85)**. These factors indicate lower customer commitment and are linked with higher churn probability.

Based on these attributes, the model predicted **Churn = 1 (Yes)** with a high probability of **0.823 (82.3%)**, suggesting strong confidence that the customer might leave the service.

The actual outcome shows that the customer **did not churn (True Churn = 0)**. This makes the prediction a **False Positive**, where the model identified the customer as high risk even though they ultimately stayed. The model correctly detected several risk related patterns, this example shows that not every high-risk profile results in actual churn.

In [192]:
row = x_test.loc[3154].to_dict()
true_label = int(y_test.loc[3154])
pred_label = int(pipe.predict(x_test.loc[[3154]])[0])
pred_prob = float(pipe.predict_proba(x_test.loc[[3154]])[:, 1][0])
    
print("Input (features):")
print(textwrap.fill(str(row), width=85))
print("\nOutput:")
print(f"  - True Churn: {true_label} (1=Yes, 0=No)")
print(f"  - Predicted Churn: {pred_label} (1=Yes, 0=No)")
print(f"  - Predicted Probability of Churn (Yes): {pred_prob:.3f}")

Input (features):
{'gender': 'Male', 'SeniorCitizen': 1, 'Partner': 'No', 'Dependents': 'No', 'tenure':
3, 'PhoneService': 'Yes', 'MultipleLines': 'Yes', 'InternetService': 'Fiber optic',
'OnlineSecurity': 'No', 'OnlineBackup': 'No', 'DeviceProtection': 'No',
'TechSupport': 'No', 'StreamingTV': 'Yes', 'StreamingMovies': 'Yes', 'Contract':
'Month-to-month', 'PaperlessBilling': 'Yes', 'PaymentMethod': 'Electronic check',
'MonthlyCharges': 94.85, 'TotalCharges': 335.75}

Output:
  - True Churn: 0 (1=Yes, 0=No)
  - Predicted Churn: 1 (1=Yes, 0=No)
  - Predicted Probability of Churn (Yes): 0.823


### Example 2 — Customer Profile (Index 2241)

This customer profile has attributes that are linked with high churn risk. The customer has a **very short tenure of 1 month**, which indicates they are a new subscriber who may not yet have developed strong commitment to the service.

The customer is also on a **month-to-month contract**, which allows them to cancel service easily without long-term commitment. Additionally, the customer uses **fiber optic internet service** and has **high monthly charges ($102.45)**. The customer pays using **electronic check**, which is another factor that has been associated with higher churn rates in this dataset.

Considering these attributes, the model predicted **Churn = 1 (Yes)** with a probability of **0.857 (85.7%)**, indicating strong confidence in the prediction. The true label confirms that the customer **did churn**, making this a **True Positive** prediction where the model correctly identified a high-risk customer.

In [193]:
row = x_test.loc[2241].to_dict()
true_label = int(y_test.loc[2241])
pred_label = int(pipe.predict(x_test.loc[[2241]])[0])
pred_prob = float(pipe.predict_proba(x_test.loc[[2241]])[:, 1][0])
    
print("Input (features):")
print(textwrap.fill(str(row), width=85))
print("\nOutput:")
print(f"  - True Churn: {true_label} (1=Yes, 0=No)")
print(f"  - Predicted Churn: {pred_label} (1=Yes, 0=No)")
print(f"  - Predicted Probability of Churn (Yes): {pred_prob:.3f}")

Input (features):
{'gender': 'Female', 'SeniorCitizen': 0, 'Partner': 'Yes', 'Dependents': 'Yes',
'tenure': 1, 'PhoneService': 'Yes', 'MultipleLines': 'Yes', 'InternetService': 'Fiber
optic', 'OnlineSecurity': 'Yes', 'OnlineBackup': 'No', 'DeviceProtection': 'Yes',
'TechSupport': 'No', 'StreamingTV': 'Yes', 'StreamingMovies': 'Yes', 'Contract':
'Month-to-month', 'PaperlessBilling': 'Yes', 'PaymentMethod': 'Electronic check',
'MonthlyCharges': 102.45, 'TotalCharges': 102.45}

Output:
  - True Churn: 1 (1=Yes, 0=No)
  - Predicted Churn: 1 (1=Yes, 0=No)
  - Predicted Probability of Churn (Yes): 0.857


### Example 3 — Customer Profile (Index 2707)

This customer in the dataset has many factors that is typically related to a lower risk of churn. The customer uses **DSL internet service** with relatively **low monthly charges ($29.35)** and pays through a **mailed check**, both of which often correspond to a basic service plan.

The customer is on a **month-to-month contract** and has a **tenure of 8 months**, the overall service cost is low. The customer also has **online security enabled**, which indicate a certain level of engagement with the service.

Based on these attributes, the model predicted **Churn = 0 (No)** with a relatively low churn probability of **0.234 (23.4%)**. The actual outcome confirms this prediction, meaning the customer **did not churn**. In this case, the model correctly identified the customer as low risk, resulting in a **True Negative** prediction.

In [194]:
row = x_test.loc[2707].to_dict()
true_label = int(y_test.loc[2707])
pred_label = int(pipe.predict(x_test.loc[[2707]])[0])
pred_prob = float(pipe.predict_proba(x_test.loc[[2707]])[:, 1][0])
    
print("Input (features):")
print(textwrap.fill(str(row), width=85))
print("\nOutput:")
print(f"  - True Churn: {true_label} (1=Yes, 0=No)")
print(f"  - Predicted Churn: {pred_label} (1=Yes, 0=No)")
print(f"  - Predicted Probability of Churn (Yes): {pred_prob:.3f}")

Input (features):
{'gender': 'Male', 'SeniorCitizen': 0, 'Partner': 'No', 'Dependents': 'No', 'tenure':
8, 'PhoneService': 'No', 'MultipleLines': 'No phone service', 'InternetService':
'DSL', 'OnlineSecurity': 'Yes', 'OnlineBackup': 'No', 'DeviceProtection': 'No',
'TechSupport': 'No', 'StreamingTV': 'No', 'StreamingMovies': 'No', 'Contract':
'Month-to-month', 'PaperlessBilling': 'No', 'PaymentMethod': 'Mailed check',
'MonthlyCharges': 29.35, 'TotalCharges': 216.45}

Output:
  - True Churn: 0 (1=Yes, 0=No)
  - Predicted Churn: 0 (1=Yes, 0=No)
  - Predicted Probability of Churn (Yes): 0.234
